# Two-stage churn pipeline (time-based CV)

**Data:** configurable — default **`data/feature_engineering/train`** (user-level tables + selected numerics from `feature_selection.py`; no re-aggregation). Set `LOAD_MODE = "preprocessed_aggregate"` to rebuild from **`data/preprocessed/train`** row-level CSVs (slower, full aggregates).

**Target (3 classes):** `invol_churn`, `vol_churn`, `not_churned`.

**Stages:**
1. Binary: `not_churned` vs `churned` (invol + vol).
2. Binary on churned users: `vol_churn` vs `invol_churn`; at inference, combine probabilities:  
   \(P(\text{not})=p_1\), \(P(\text{vol})=(1-p_1)\cdot p_2\), \(P(\text{invol})=(1-p_1)\cdot(1-p_2)\).

**Validation:** `TimeSeriesSplit` on users sorted by `subscription_start_date` (no random shuffled CV).

**Metrics (0–100 scale):** **Composed** binary churn vs `not_churned` (from the two-stage probabilities). **Per stage:** stage 1 = churn vs stay (accuracy, F1 for churn, ROC-AUC); stage 2 = vol vs invol on validation users who churned only (accuracy, F1 for vol, ROC-AUC).

**Environment:** `/home/ansar/work/.venv`


In [1]:
from __future__ import annotations

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
N_TS_SPLITS = 5

PROJECT_ROOT = Path("/home/ansar/work/hack-nu-26")
PREPROCESSED_TRAIN = PROJECT_ROOT / "data/preprocessed/train"
FE_TRAIN = PROJECT_ROOT / "data/feature_engineering/train"

# "feature_engineering" — load merged user-level CSVs from feature_selection output (fast).
# "preprocessed_aggregate" — rebuild aggregates from raw preprocessed row-level tables (slow).
LOAD_MODE: str = "feature_engineering"

# Binary churn vs not_churned uses y_binary_stay / composed P(churn) from the two-stage head.


## 1. Load tables and build one row per user

**`LOAD_MODE == "feature_engineering"`:** reads `train_users*.csv` under `data/feature_engineering/train` (properties include `subscription_start_ts`; selected numerics in `train_users_selected_numerics.csv`). No chunked generation scan.

**`LOAD_MODE == "preprocessed_aggregate"`:** same as before — join purchases ↔ attempts, aggregate generations from `data/preprocessed/train`. `train_users_transaction_attempts.csv` has no `user_id`; rows join via `transaction_id`.


In [ ]:
def load_user_frame_feature_engineering(fe_train: Path) -> pd.DataFrame:
    """One row per user from feature_engineering split tables (after feature_selection)."""
    u = pd.read_csv(fe_train / "train_users.csv", usecols=["user_id", "churn_status"])
    prop = pd.read_csv(
        fe_train / "train_users_properties.csv",
        usecols=["user_id", "subscription_plan", "country_code", "subscription_start_ts"],
    )
    q = pd.read_csv(
        fe_train / "train_users_quizzes.csv",
        usecols=[
            "user_id",
            "source",
            "team_size",
            "experience",
            "usage_plan",
            "frustration",
            "first_feature",
            "role",
        ],
    )
    u = u.drop_duplicates(subset=["user_id"], keep="first")
    prop = prop.drop_duplicates(subset=["user_id"], keep="first")
    q = q.drop_duplicates(subset=["user_id"], keep="first")
    num = pd.read_csv(fe_train / "train_users_selected_numerics.csv")
    drop_blank = [c for c in num.columns if c == "" or str(c).startswith("Unnamed")]
    if drop_blank:
        num = num.drop(columns=drop_blank)
    num = num.drop_duplicates(subset=["user_id"], keep="first")
    df = u.merge(prop, on="user_id", how="left").merge(q, on="user_id", how="left")
    num_only = [c for c in num.columns if c != "user_id"]
    df = df.merge(num[["user_id"] + num_only], on="user_id", how="left")
    return df


def load_user_frame_preprocessed_aggregate(raw_dir: Path) -> pd.DataFrame:
    """Rebuild user-level aggregates from raw preprocessed train CSVs."""

    def load_base_users() -> pd.DataFrame:
        u = pd.read_csv(raw_dir / "train_users.csv", usecols=["user_id", "churn_status"])
        prop = pd.read_csv(
            raw_dir / "train_users_properties.csv",
            usecols=["user_id", "subscription_start_date", "subscription_plan", "country_code"],
        )
        q = pd.read_csv(
            raw_dir / "train_users_quizzes.csv",
            usecols=[
                "user_id",
                "source",
                "team_size",
                "experience",
                "usage_plan",
                "frustration",
                "first_feature",
                "role",
            ],
        )
        df = u.merge(prop, on="user_id", how="left").merge(q, on="user_id", how="left")
        df["subscription_start_ts"] = pd.to_datetime(
            df["subscription_start_date"], utc=True, errors="coerce"
        ).astype("int64")
        return df.drop(columns=["subscription_start_date"])

    def aggregate_purchases_attempts() -> pd.DataFrame:
        pur = pd.read_csv(
            raw_dir / "train_users_purchases.csv",
            usecols=["user_id", "transaction_id", "purchase_type", "purchase_amount_dollars"],
            low_memory=False,
        )
        ta_use = [
            "transaction_id",
            "amount_in_usd",
            "billing_address_country",
            "card_3d_secure_support",
            "card_country",
            "card_funding",
            "cvc_check",
            "digital_wallet",
            "is_3d_secure",
            "is_3d_secure_authenticated",
            "payment_method_type",
            "bank_country",
            "is_prepaid",
            "is_virtual",
            "is_business",
        ]
        ta = pd.read_csv(
            raw_dir / "train_users_transaction_attempts.csv",
            usecols=ta_use,
            low_memory=False,
        )
        for c in ta.columns:
            if c.startswith("is_"):
                ta[c] = ta[c].map(lambda x: str(x).lower() in {"true", "1"})
        m = pur.merge(ta, on="transaction_id", how="left", suffixes=("_pur", ""))
        bool_cols = [c for c in ta.columns if c.startswith("is_")]
        if bool_cols:
            m["att_card_flags_row"] = m[bool_cols].astype(np.float64).mean(axis=1)
        else:
            m["att_card_flags_row"] = 0.0
        mix_cols = [
            c
            for c in [
                "billing_address_country",
                "card_country",
                "card_funding",
                "payment_method_type",
                "cvc_check",
                "digital_wallet",
                "card_3d_secure_support",
                "bank_country",
            ]
            if c in m.columns
        ]
        if mix_cols:
            m["att_payment_mix_key"] = m[mix_cols].astype(str).apply(
                lambda r: "|".join(r.values), axis=1
            )
        else:
            m["att_payment_mix_key"] = ""
        agg_dict = {
            "transaction_id": "count",
            "purchase_amount_dollars": "sum",
            "purchase_type": "nunique",
            "amount_in_usd": "sum",
            "att_card_flags_row": "mean",
            "att_payment_mix_key": "nunique",
        }
        g = m.groupby("user_id", as_index=False).agg(agg_dict)
        return g.rename(
            columns={
                "transaction_id": "purch_n",
                "purchase_amount_dollars": "purch_amount_sum",
                "purchase_type": "purch_type_nunique",
                "amount_in_usd": "att_amount_sum",
                "att_card_flags_row": "att_card_flags_mean",
                "att_payment_mix_key": "att_payment_mix_nunique",
            }
        )

    def aggregate_generations(chunksize: int = 2_000_000) -> pd.DataFrame:
        path = raw_dir / "train_users_generations.csv"
        usecols = ["user_id", "status", "generation_type"]
        status_parts: list[pd.Series] = []
        type_parts: list[pd.Series] = []
        for chunk in pd.read_csv(path, chunksize=chunksize, usecols=usecols):
            status_parts.append(chunk.groupby(["user_id", "status"]).size())
            type_parts.append(chunk.groupby(["user_id", "generation_type"]).size())
        st = pd.concat(status_parts).groupby(level=[0, 1]).sum().unstack(fill_value=0)
        st.columns = [f"gen_status_{c}" for c in st.columns.astype(str)]
        gt = pd.concat(type_parts).groupby(level=[0, 1]).sum().unstack(fill_value=0)
        gt.columns = [f"gen_type_{c}" for c in gt.columns.astype(str)]
        out = st.join(gt, how="outer").fillna(0).astype(np.float32)
        out["gen_total"] = out[[c for c in out.columns if c.startswith("gen_status_")]].sum(axis=1)
        return out.reset_index()

    df = load_base_users()
    pa = aggregate_purchases_attempts()
    df = df.merge(pa, on="user_id", how="left")
    gen = aggregate_generations()
    return df.merge(gen, on="user_id", how="left")


if LOAD_MODE == "feature_engineering":
    df = load_user_frame_feature_engineering(FE_TRAIN)
elif LOAD_MODE == "preprocessed_aggregate":
    df = load_user_frame_preprocessed_aggregate(PREPROCESSED_TRAIN)
else:
    raise ValueError(f"Unknown LOAD_MODE: {LOAD_MODE!r}")

num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in df.columns if c not in num_cols and c not in {"user_id", "churn_status"}]
for c in cat_cols:
    df[c] = df[c].fillna("skipped").astype(str)

df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
df = df.sort_values("subscription_start_ts").reset_index(drop=True)

print(f"LOAD_MODE={LOAD_MODE!r}", df.shape, "numeric:", len(num_cols), "categorical:", len(cat_cols))
df[["user_id", "churn_status", "subscription_start_ts"]].head()


## 2. Targets, matrices, preprocessing (OrdinalEncoder + imputation + optional scaling)

In [ ]:
y_binary_stay = (df["churn_status"] == "not_churned").astype(np.int8).values  # 1 = not churned
y_churn_subtype = (df["churn_status"] == "vol_churn").astype(np.int8).values  # 1 = vol among all users

feature_cols = [c for c in df.columns if c not in {"user_id", "churn_status"}]
X_df = df[feature_cols].copy()

num_features = [c for c in feature_cols if c in num_cols]
cat_features = [c for c in feature_cols if c in cat_cols]

for c in cat_features:
    X_df[c] = (
        pd.Series(X_df[c], dtype=object).fillna("skipped").astype(str).astype(object)
    )


def make_preprocessor(scale: bool) -> ColumnTransformer:
    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
        + ([("scaler", StandardScaler())] if scale else [])
    )
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_features),
            ("cat", cat_pipe, cat_features),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def hierarchical_class_proba(p_stay: np.ndarray, p_vol_if_churn: np.ndarray) -> np.ndarray:
    """p_stay = P(not_churned); p_vol_if_churn = P(vol_churn | churned). Columns: invol, vol, not."""
    p_stay = np.clip(p_stay, 1e-8, 1 - 1e-8)
    p_vol_if_churn = np.clip(p_vol_if_churn, 1e-8, 1 - 1e-8)
    p_churn = 1.0 - p_stay
    p_vol = p_churn * p_vol_if_churn
    p_invol = p_churn * (1.0 - p_vol_if_churn)
    return np.column_stack([p_invol, p_vol, p_stay])


def metrics_binary_churn_pct(y_stay_val: np.ndarray, proba3: np.ndarray) -> dict[str, float]:
    """y_stay_val: 1 = not_churned, 0 = churned (vol+invol). proba3 columns: invol, vol, not."""
    y_churn = (y_stay_val == 0).astype(np.int8)
    p_churn = proba3[:, 0] + proba3[:, 1]
    pred_churn = (p_churn >= 0.5).astype(np.int8)
    return {
        "accuracy_pct": float(100.0 * accuracy_score(y_churn, pred_churn)),
        "f1_churn_pct": float(
            100.0 * f1_score(y_churn, pred_churn, pos_label=1, zero_division=0)
        ),
        "roc_auc_churn_pct": float(100.0 * roc_auc_score(y_churn, p_churn)),
    }

## 3. Time-based CV + model factories

Models: **XGBoost**, **CatBoost** (native categoricals), **RandomForest**, **soft Voting** (XGB + CatBoost on ordinal matrix), **MLP**, **TabNet**.

For each fold and model, **`met`** = composed binary churn vs not (from the two-stage head). **`per_stage`** table = **stage 1** (churn vs stay: accuracy, F1 for churn, ROC-AUC for `P(churn)`) and **stage 2** on validation churned users only (vol vs invol: accuracy, F1 for vol, ROC-AUC for `P(vol)`). All in **0–100**.

In [ ]:
import torch
from sklearn.base import BaseEstimator, ClassifierMixin
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
from xgboost import XGBClassifier


def _array2d_for_tabnet(X) -> np.ndarray:
    """TabNet expects a dense ndarray; preprocessor may return ndarray or sparse."""
    if hasattr(X, "values"):
        X = X.values
    if hasattr(X, "toarray"):
        X = X.toarray()
    return np.asarray(X, dtype=np.float64)


def _binary_cls_metrics_pct(y: np.ndarray, p_pos: np.ndarray) -> tuple[float, float, float]:
    """y in 0/1, p_pos = P(y=1); threshold 0.5. Returns (accuracy, f1 for class 1, roc_auc) in 0-100."""
    y = np.asarray(y, dtype=np.int8)
    p = np.clip(np.asarray(p_pos, dtype=np.float64), 1e-8, 1.0 - 1e-8)
    pred = (p >= 0.5).astype(np.int8)
    acc = 100.0 * float(accuracy_score(y, pred))
    f1v = 100.0 * float(f1_score(y, pred, pos_label=1, zero_division=0))
    if np.unique(y).size < 2:
        auc = float("nan")
    else:
        auc = 100.0 * float(roc_auc_score(y, p))
    return acc, f1v, auc


class CatBoostOrdinalEnsembleBlock(ClassifierMixin, BaseEstimator):
    """CatBoost on the ordinal-encoded design matrix; VotingClassifier cannot pass cat_features.

    ClassifierMixin must be listed before BaseEstimator so sklearn tags report a classifier.
    """

    def __init__(
        self,
        cat_column_indices,
        iterations: int = 400,
        depth: int = 6,
        learning_rate: float = 0.05,
        random_seed: int = 0,
    ):
        self.cat_column_indices = cat_column_indices
        self.iterations = iterations
        self.depth = depth
        self.learning_rate = learning_rate
        self.random_seed = random_seed

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.estimator_type = "classifier"
        return tags

    def fit(self, X, y):
        self.model_ = CatBoostClassifier(
            loss_function="Logloss",
            iterations=self.iterations,
            depth=self.depth,
            learning_rate=self.learning_rate,
            random_seed=self.random_seed,
            verbose=False,
            allow_writing_files=False,
        )
        # X is ordinal-encoded floats from ColumnTransformer — no raw categoricals for CatBoost.
        self.model_.fit(X, y, cat_features=[])
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)


def fit_predict_hierarchical(
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    y_stay_train: np.ndarray,
    y_stay_val: np.ndarray,
    y_vol_train_all: np.ndarray,
    y_vol_val_all: np.ndarray,
    churned_train_mask: np.ndarray,
    mode: str,
) -> tuple[np.ndarray, dict[str, float]]:
    """Returns (val proba n×3, stage diagnostics)."""
    diag: dict[str, float] = {}

    if mode == "catboost":
        X_train = X_train.copy()
        X_val = X_val.copy()
        for c in cat_features:
            X_train[c] = X_train[c].fillna("skipped").astype(str).astype(object)
            X_val[c] = X_val[c].fillna("skipped").astype(str).astype(object)
        cat_idx = [X_train.columns.get_loc(c) for c in cat_features]
        m1 = CatBoostClassifier(
            loss_function="Logloss",
            iterations=400,
            depth=6,
            learning_rate=0.05,
            random_seed=RANDOM_STATE,
            verbose=False,
            allow_writing_files=False,
        )
        m1.fit(X_train, y_stay_train, cat_features=cat_idx)
        p_stay_val = m1.predict_proba(X_val)[:, 1]
        X_tr2 = X_train.iloc[churned_train_mask]
        y_tr2 = y_vol_train_all[churned_train_mask]
        m2 = CatBoostClassifier(
            loss_function="Logloss",
            iterations=400,
            depth=6,
            learning_rate=0.05,
            random_seed=RANDOM_STATE + 1,
            verbose=False,
            allow_writing_files=False,
        )
        m2.fit(X_tr2, y_tr2, cat_features=cat_idx)
        p_vol_if_churn_val = m2.predict_proba(X_val)[:, 1]

    elif mode in {"xgb", "rf", "mlp", "voting", "tabnet"}:
        scale = mode == "mlp" or mode == "tabnet"
        pre = make_preprocessor(scale=scale)
        Xtr = pre.fit_transform(X_train)
        Xva = pre.transform(X_val)

        if mode == "xgb":
            m1 = XGBClassifier(
                n_estimators=400,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                eval_metric="logloss",
                verbosity=0,
            )
            m1.fit(Xtr, y_stay_train)
            p_stay_val = m1.predict_proba(Xva)[:, 1]
            m2 = XGBClassifier(
                n_estimators=400,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.85,
                reg_lambda=1.0,
                random_state=RANDOM_STATE + 1,
                n_jobs=-1,
                eval_metric="logloss",
                verbosity=0,
            )
            m2.fit(Xtr[churned_train_mask], y_vol_train_all[churned_train_mask])
            p_vol_if_churn_val = m2.predict_proba(Xva)[:, 1]

        elif mode == "rf":
            m1 = RandomForestClassifier(
                n_estimators=300,
                max_depth=16,
                min_samples_leaf=4,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )
            m1.fit(Xtr, y_stay_train)
            p_stay_val = m1.predict_proba(Xva)[:, 1]
            m2 = RandomForestClassifier(
                n_estimators=300,
                max_depth=16,
                min_samples_leaf=4,
                random_state=RANDOM_STATE + 1,
                n_jobs=-1,
            )
            m2.fit(Xtr[churned_train_mask], y_vol_train_all[churned_train_mask])
            p_vol_if_churn_val = m2.predict_proba(Xva)[:, 1]

        elif mode == "mlp":
            m1 = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-4,
                batch_size=512,
                learning_rate_init=1e-3,
                max_iter=80,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=8,
                random_state=RANDOM_STATE,
            )
            m1.fit(Xtr, y_stay_train)
            p_stay_val = m1.predict_proba(Xva)[:, 1]
            m2 = MLPClassifier(
                hidden_layer_sizes=(128, 64),
                activation="relu",
                alpha=1e-4,
                batch_size=256,
                learning_rate_init=1e-3,
                max_iter=120,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
                random_state=RANDOM_STATE + 1,
            )
            m2.fit(Xtr[churned_train_mask], y_vol_train_all[churned_train_mask])
            p_vol_if_churn_val = m2.predict_proba(Xva)[:, 1]

        elif mode == "voting":
            n_num = len(num_features)
            cat_idx_ord = list(range(n_num, n_num + len(cat_features)))
            x1 = XGBClassifier(
                n_estimators=250,
                max_depth=6,
                learning_rate=0.06,
                subsample=0.9,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                eval_metric="logloss",
                verbosity=0,
            )
            cb1 = CatBoostOrdinalEnsembleBlock(
                cat_column_indices=cat_idx_ord,
                iterations=400,
                depth=6,
                learning_rate=0.05,
                random_seed=RANDOM_STATE + 3,
            )
            m1 = VotingClassifier(
                estimators=[("xgb", x1), ("catboost", cb1)],
                voting="soft",
                weights=[1.0, 1.0],
            )
            m1.fit(Xtr, y_stay_train)
            p_stay_val = m1.predict_proba(Xva)[:, 1]
            x2 = XGBClassifier(
                n_estimators=250,
                max_depth=6,
                learning_rate=0.06,
                subsample=0.9,
                random_state=RANDOM_STATE + 7,
                n_jobs=-1,
                eval_metric="logloss",
                verbosity=0,
            )
            cb2 = CatBoostOrdinalEnsembleBlock(
                cat_column_indices=cat_idx_ord,
                iterations=400,
                depth=6,
                learning_rate=0.05,
                random_seed=RANDOM_STATE + 11,
            )
            m2 = VotingClassifier(
                estimators=[("xgb", x2), ("catboost", cb2)],
                voting="soft",
                weights=[1.0, 1.0],
            )
            m2.fit(Xtr[churned_train_mask], y_vol_train_all[churned_train_mask])
            p_vol_if_churn_val = m2.predict_proba(Xva)[:, 1]

        else:  # tabnet
            m1 = TabNetClassifier(
                verbose=0,
                seed=RANDOM_STATE,
                n_d=16,
                n_a=16,
                n_steps=3,
                gamma=1.5,
                lambda_sparse=1e-4,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=2e-2),
                mask_type="sparsemax",
            )
            n1 = len(Xtr)
            split = int(max(0.15 * n1, 1))
            m1.fit(
                Xtr[:-split],
                y_stay_train[:-split],
                eval_set=[(Xtr[-split:], y_stay_train[-split:])],
                eval_metric=["auc"],
                max_epochs=120,
                patience=15,
                batch_size=2048,
                virtual_batch_size=128,
            )
            Xva_tn = _array2d_for_tabnet(Xva)
            p_stay_val = m1.predict_proba(Xva_tn)[:, 1]
            Xtr_c = Xtr[churned_train_mask]
            ytr_c = y_vol_train_all[churned_train_mask]
            m2 = TabNetClassifier(
                verbose=0,
                seed=RANDOM_STATE + 1,
                n_d=16,
                n_a=16,
                n_steps=3,
                gamma=1.5,
                lambda_sparse=1e-4,
                optimizer_fn=torch.optim.Adam,
                optimizer_params=dict(lr=2e-2),
                mask_type="sparsemax",
            )
            n2 = len(Xtr_c)
            split2 = int(max(0.15 * n2, 1))
            m2.fit(
                Xtr_c[:-split2],
                ytr_c[:-split2],
                eval_set=[(Xtr_c[-split2:], ytr_c[-split2:])],
                eval_metric=["auc"],
                max_epochs=120,
                patience=15,
                batch_size=min(2048, max(256, n2 // 4)),
                virtual_batch_size=128,
            )
            p_vol_if_churn_val = m2.predict_proba(Xva_tn)[:, 1]
    else:
        raise ValueError(mode)

    # Per-stage binary metrics (0-100): stage1 churn vs stay; stage2 vol vs invol on val churned only
    y_churn_val = (y_stay_val == 0).astype(np.int8)
    p_churn_val = 1.0 - np.asarray(p_stay_val, dtype=np.float64)
    s1_acc, s1_f1, s1_auc = _binary_cls_metrics_pct(y_churn_val, p_churn_val)
    diag["stage1_accuracy_pct"] = s1_acc
    diag["stage1_f1_churn_pct"] = s1_f1
    diag["stage1_roc_auc_churn_pct"] = s1_auc
    mask_churn_val = y_stay_val == 0
    if mask_churn_val.sum() > 0:
        yv = y_vol_val_all[mask_churn_val].astype(np.int8)
        pv = p_vol_if_churn_val[mask_churn_val]
        s2_acc, s2_f1, s2_auc = _binary_cls_metrics_pct(yv, pv)
        diag["stage2_accuracy_pct"] = s2_acc
        diag["stage2_f1_vol_pct"] = s2_f1
        diag["stage2_roc_auc_vol_pct"] = s2_auc
    else:
        diag["stage2_accuracy_pct"] = float("nan")
        diag["stage2_f1_vol_pct"] = float("nan")
        diag["stage2_roc_auc_vol_pct"] = float("nan")

    proba3 = hierarchical_class_proba(p_stay_val, p_vol_if_churn_val)
    return proba3, diag


def run_time_series_cv(mode: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    tss = TimeSeriesSplit(n_splits=N_TS_SPLITS)
    fold_rows = []
    diag_rows = []
    for fold, (tr, va) in enumerate(tss.split(X_df)):
        X_train, X_val = X_df.iloc[tr], X_df.iloc[va]
        ys_tr, ys_va = y_binary_stay[tr], y_binary_stay[va]
        yv_tr, yv_va = y_churn_subtype[tr], y_churn_subtype[va]
        churned_tr = df["churn_status"].iloc[tr].ne("not_churned").values
        proba, diag = fit_predict_hierarchical(
            X_train,
            X_val,
            ys_tr,
            ys_va,
            yv_tr,
            yv_va,
            churned_tr,
            mode=mode,
        )
        m = metrics_binary_churn_pct(ys_va, proba)
        m["fold"] = fold
        fold_rows.append(m)
        d = {"fold": fold, **diag}
        diag_rows.append(d)
    return pd.DataFrame(fold_rows), pd.DataFrame(diag_rows)


MODELS = ["xgb", "catboost", "rf", "voting", "mlp", "tabnet"]
summary = []
for name in MODELS:
    print("===", name, "===")
    met, diag = run_time_series_cv(name)
    met["model"] = name
    summary.append(met)
    display(met)
    per_stage = diag[
        [
            "fold",
            "stage1_accuracy_pct",
            "stage1_f1_churn_pct",
            "stage1_roc_auc_churn_pct",
            "stage2_accuracy_pct",
            "stage2_f1_vol_pct",
            "stage2_roc_auc_vol_pct",
        ]
    ]
    display(per_stage)

summary_df = pd.concat(summary, ignore_index=True)
agg = summary_df.groupby("model")[
    ["accuracy_pct", "f1_churn_pct", "roc_auc_churn_pct"]
].agg(["mean", "std"])
display(agg)

=== xgb ===


,accuracy_pct,f1_churn_pct,roc_auc_churn_pct,fold,model
0,62.580000,60.530202,67.540374,0,xgb
1,61.006667,57.887537,65.750951,1,xgb
2,61.140000,62.983425,66.423608,2,xgb
3,58.733333,53.020644,63.371148,3,xgb
4,58.940000,54.265984,62.763048,4,xgb


,fold,stage1_accuracy_pct,stage1_f1_churn_pct,stage1_roc_auc_churn_pct,stage2_accuracy_pct,stage2_f1_vol_pct,stage2_roc_auc_vol_pct
0,0,62.580000,60.530202,67.540376,67.936381,45.240813,72.170553
1,1,61.006667,57.887537,65.750951,52.164646,23.121387,63.978277
2,2,61.140000,62.983425,66.423610,61.430669,66.403162,68.975650
3,3,58.733333,53.020644,63.371146,65.464796,63.057504,70.671936
4,4,58.940000,54.265984,62.763045,65.468602,66.393875,70.544152


=== catboost ===


,accuracy_pct,f1_churn_pct,roc_auc_churn_pct,fold,model
0,64.620000,59.361360,70.172679,0,catboost
1,62.600000,58.561087,67.933405,1,catboost
2,61.540000,62.271925,67.188558,2,catboost
3,59.366667,54.828430,64.181702,3,catboost
4,59.400000,54.808549,63.618083,4,catboost


...

=== rf ===


,accuracy_pct,f1_churn_pct,roc_auc_churn_pct,fold,model
0,62.886667,59.385715,68.049917,0,rf
1,59.940000,51.395292,64.722826,1,rf
2,60.453333,59.252645,65.197965,2,rf
3,58.066667,51.852419,62.211882,3,rf
4,58.406667,53.367217,62.157384,4,rf


,fold,stage1_accuracy_pct,stage1_f1_churn_pct,stage1_roc_auc_churn_pct,stage2_accuracy_pct,stage2_f1_vol_pct,stage2_roc_auc_vol_pct
0,0,62.886667,59.385715,68.049917,65.316760,56.481637,70.705734
1,1,59.940000,51.395292,64.722826,47.888637,4.538799,59.628868
2,2,60.453333,59.252645,65.197965,63.419191,63.370306,68.746881
3,3,58.066667,51.852419,62.211840,62.383637,56.298553,67.166348
4,4,58.406667,53.367217,62.157384,62.145667,60.304366,66.864827


=== voting ===


ValueError: The estimator CatBoostOrdinalEnsembleBlock should be a classifier.